# 3. Exploratory Data Analysis

### 3.1 Import Libraries & Load Data

In [2]:
import pandas as pd
from pathlib import Path

# Define project root
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

# Define processed data directory
data_dir = project_root / "data" / "processed"

# Load cleaned datasets and parse date columns
activity_df = pd.read_csv(
    data_dir / "daily_activity_cleaned.csv",
    parse_dates=["ActivityDate"]
)

sleep_df = pd.read_csv(
    data_dir / "sleep_cleaned.csv",
    parse_dates=["SleepDay"]
)

# Display dataset shapes
print("Daily Activity shape:", activity_df.shape)
print("Sleep Data shape:", sleep_df.shape)

# Verify date column data types
print("\nActivityDate type:", activity_df["ActivityDate"].dtype)
print("SleepDay type:", sleep_df["SleepDay"].dtype)


Daily Activity shape: (940, 15)
Sleep Data shape: (410, 5)

ActivityDate type: datetime64[us]
SleepDay type: datetime64[us]


### 3.2 Descriptive Statistics

In [3]:
# Calculate descriptive statistics for Daily Activity
activity_stats = (
    activity_df
    .drop(columns=["Id", "ActivityDate"])
    .describe()
    .T
    .round(2)
)

print("Daily Activity - Descriptive Statistics:")
display(activity_stats)

# Calculate descriptive statistics for Sleep Data
sleep_stats = (
    sleep_df
    .drop(columns=["Id", "SleepDay"])
    .describe()
    .T
    .round(2)
)

print("\nSleep Data - Descriptive Statistics:")
display(sleep_stats)

Daily Activity - Descriptive Statistics:


,count,mean,std,min,25%,50%,75%,max
TotalSteps,940.0,7637.91,5087.15,0.0,3789.75,7405.50,10727.00,36019.00
TotalDistance,940.0,5.49,3.92,0.0,2.62,5.24,7.71,28.03
TrackerDistance,940.0,5.48,3.91,0.0,2.62,5.24,7.71,28.03
LoggedActivitiesDistance,940.0,0.11,0.62,0.0,0.00,0.00,0.00,4.94
VeryActiveDistance,940.0,1.50,2.66,0.0,0.00,0.21,2.05,21.92
ModeratelyActiveDistance,940.0,0.57,0.88,0.0,0.00,0.24,0.80,6.48
LightActiveDistance,940.0,3.34,2.04,0.0,1.95,3.36,4.78,10.71
SedentaryActiveDistance,940.0,0.00,0.01,0.0,0.00,0.00,0.00,0.11
VeryActiveMinutes,940.0,21.16,32.84,0.0,0.00,4.00,32.00,210.00
FairlyActiveMinutes,940.0,13.56,19.99,0.0,0.00,6.00,19.00,143.00



Sleep Data - Descriptive Statistics:


,count,mean,std,min,25%,50%,75%,max
TotalSleepRecords,410.0,1.12,0.35,1.0,1.00,1.0,1.0,3.0
TotalMinutesAsleep,410.0,419.17,118.64,58.0,361.00,432.5,490.0,796.0
TotalTimeInBed,410.0,458.48,127.46,61.0,403.75,463.0,526.0,961.0


### 3.3 Daily Activity Analysis

#### 3.3.1 Average Daily Steps per User

In [4]:
# Calculate average daily steps and recorded days for each user
user_steps = (
    activity_df.groupby("Id")
    .agg(
        RecordedDays=("ActivityDate", "nunique"),
        AverageDailySteps=("TotalSteps", "mean")
    )
    .reset_index()
)

# Compare daily-record and user-level averages
print("Average steps per daily record:",
      round(activity_df["TotalSteps"].mean(), 2))

print("Average steps across users:",
      round(user_steps["AverageDailySteps"].mean(), 2))

# Display users with the highest average daily steps
user_steps = user_steps.sort_values(
    "AverageDailySteps", ascending=False
)

display(user_steps.head(10).round(2))


Average steps per daily record: 7637.91
Average steps across users: 7519.27


,Id,RecordedDays,AverageDailySteps
32,8877689391,31,16040.03
27,8053475328,31,14763.29
0,1503960366,31,12116.74
5,2022484408,31,11370.65
25,7007744171,26,11323.42
11,3977333714,30,10984.57
15,4388161847,31,10813.94
24,6962181067,31,9794.81
8,2347167796,18,9519.67
26,7086361926,31,9371.77


#### 3.3.2 Daily Steps Distribution

In [6]:
# Examine daily steps distribution
steps = activity_df["TotalSteps"]

print("Minimum Steps:", steps.min())
print("Median Steps:", steps.median())
print("Maximum Steps:", steps.max())
print("25th Percentile:", steps.quantile(0.25))
print("75th Percentile:", steps.quantile(0.75))
print("Zero-Step Days:", (steps == 0).sum())


Minimum Steps: 0
Median Steps: 7405.5
Maximum Steps: 36019
25th Percentile: 3789.75
75th Percentile: 10727.0
Zero-Step Days: 77


#### 3.3.3 Activity Intensity Analysis

In [7]:
# Select active minute columns
active_minutes = activity_df[
    ["VeryActiveMinutes", "FairlyActiveMinutes", "LightlyActiveMinutes"]
]

# Calculate average daily minutes and share of active minutes
intensity_summary = pd.DataFrame({
    "Average Daily Minutes": active_minutes.mean().round(2),
    "Share of Active Minutes (%)": (
        active_minutes.sum() / active_minutes.sum().sum() * 100
    ).round(2)
})

display(intensity_summary)

# Count days without very active minutes
print("Days with zero very active minutes:",
      (activity_df["VeryActiveMinutes"] == 0).sum())


,Average Daily Minutes,Share of Active Minutes (%)
VeryActiveMinutes,21.16,9.30
FairlyActiveMinutes,13.56,5.96
LightlyActiveMinutes,192.81,84.74


Days with zero very active minutes: 409


#### 3.3.4 Sedentary Time Analysis

In [8]:
# Select sedentary minutes
sedentary = activity_df["SedentaryMinutes"]

# Count records with 12 or more sedentary hours
high_sedentary_days = (sedentary >= 720).sum()

# Calculate the percentage of records
high_sedentary_pct = high_sedentary_days / len(sedentary) * 100

# Count records with zero sedentary minutes
zero_sedentary_days = (sedentary == 0).sum()

# Count records with 24 hours of sedentary minutes
full_day_sedentary = (sedentary == 1440).sum()

print("Records with 12+ sedentary hours:", high_sedentary_days)
print("Percentage of records:", round(high_sedentary_pct, 2), "%")
print("Records with zero sedentary minutes:", zero_sedentary_days)
print("Records with 1440 sedentary minutes:", full_day_sedentary)


Records with 12+ sedentary hours: 727
Percentage of records: 77.34 %
Records with zero sedentary minutes: 1
Records with 1440 sedentary minutes: 79


#### 3.3.5 Activity Levels Across Users

In [9]:
# Calculate average activity minutes for each user
user_activity = (
    activity_df.groupby("Id")
    .agg(
        RecordedDays=("ActivityDate", "nunique"),
        AverageVeryActive=("VeryActiveMinutes", "mean"),
        AverageFairlyActive=("FairlyActiveMinutes", "mean"),
        AverageLightlyActive=("LightlyActiveMinutes", "mean"),
        AverageSedentary=("SedentaryMinutes", "mean")
    )
    .reset_index()
)

# Calculate total average active minutes per user
user_activity["AverageTotalActive"] = (
    user_activity["AverageVeryActive"]
    + user_activity["AverageFairlyActive"]
    + user_activity["AverageLightlyActive"]
)

# Sort users by average active minutes
user_activity = user_activity.sort_values(
    "AverageTotalActive", ascending=False
)

display(user_activity.head(10).round(2))


,Id,RecordedDays,AverageVeryActive,AverageFairlyActive,AverageLightlyActive,AverageSedentary,AverageTotalActive
10,3372868164,20,9.15,4.10,327.90,1077.55,341.15
9,2873212765,31,14.10,6.13,308.00,1097.19,328.23
25,7007744171,26,31.04,16.27,280.73,1055.35,328.04
5,2022484408,31,36.29,19.35,257.45,1112.58,313.10
32,8877689391,31,66.06,9.94,234.71,1112.87,310.71
17,4558609924,31,10.39,13.71,284.97,1093.61,309.06
21,6117666160,28,1.57,2.04,288.36,796.29,291.96
24,6962181067,31,22.81,18.52,245.81,662.32,287.13
8,2347167796,18,13.50,20.56,252.50,687.17,286.56
0,1503960366,31,38.71,19.16,219.94,848.16,277.81


#### 3.3.6 Steps & Calories Relationship

In [10]:
# Calculate Pearson correlation between steps and calories
steps_calories_corr = activity_df["TotalSteps"].corr(
    activity_df["Calories"],
    method="pearson"
)

print("Steps & Calories Pearson Correlation:",
      round(steps_calories_corr, 3))


Steps & Calories Pearson Correlation: 0.592
